## Objective

Analyze campaign performance across customer segments, customer type,
acquisition channel, and region for the 2025 in-scope campaign population,
to help Marketing understand which segments participate most, generate
the most value, and where campaign effectiveness differs for new vs.
existing customers.

### Grain
One row per (campaign_id, customer_id), for customers who were eligible
for at least one promotion in an in-scope 2025 campaign.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW campaign_scope AS
SELECT campaign_id, campaign_name, campaign_type, team, start_date, end_date
FROM `campaign&promotion`.gold.campaign_performance;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_campaign_participation AS
SELECT
    campaign_id,
    customer_id,
    MAX(CAST(eligible_flag AS INT)) = 1 AS eligible_flag,
    MAX(CAST(participated_flag AS INT)) = 1 AS participated_flag,
    MIN(participation_date) AS first_participation_date
FROM `campaign&promotion`.gold.fct_campaign_participation
WHERE campaign_id IN (SELECT campaign_id FROM campaign_scope)
GROUP BY campaign_id, customer_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_campaign_transactions AS
SELECT
    p.campaign_id,
    p.customer_id,
    COUNT(DISTINCT t.transaction_id) AS transaction_count,
    SUM(t.amount) AS transaction_value
FROM customer_campaign_participation p
JOIN campaign_scope c
    ON p.campaign_id = c.campaign_id
JOIN `campaign&promotion`.gold.fct_transaction t
    ON p.customer_id = t.customer_id
   AND t.transaction_date BETWEEN c.start_date AND c.end_date
   AND t.status = 'SUCCESS'
WHERE p.participated_flag = TRUE
GROUP BY p.campaign_id, p.customer_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_campaign_rewards AS
SELECT
    campaign_id,
    customer_id,
    COUNT(DISTINCT redemption_id) AS redemption_count,
    SUM(reward_amount) AS reward_cost
FROM `campaign&promotion`.gold.fct_promotion_redemption
WHERE campaign_id IN (SELECT campaign_id FROM campaign_scope)
GROUP BY campaign_id, customer_id;

In [0]:
%sql

CREATE OR REPLACE TABLE `campaign&promotion`.gold.customer_campaign_performance AS

SELECT
    p.campaign_id,
    c.campaign_name,
    c.campaign_type,
    c.team,

    p.customer_id,
    d.customer_segment,
    d.customer_type,
    d.acquisition_channel,
    d.region,
    d.city,
    d.registration_date,

    p.eligible_flag,
    p.participated_flag,

    CASE
        WHEN d.registration_date BETWEEN c.start_date AND c.end_date
        THEN TRUE ELSE FALSE
    END AS is_new_customer,

    COALESCE(t.transaction_count, 0) AS transaction_count,
    COALESCE(t.transaction_value, 0) AS transaction_value,
    COALESCE(r.redemption_count, 0) AS redemption_count,
    COALESCE(r.reward_cost, 0) AS reward_cost

FROM customer_campaign_participation p

JOIN campaign_scope c
    ON p.campaign_id = c.campaign_id

JOIN `campaign&promotion`.gold.dim_customer d
    ON p.customer_id = d.customer_id

LEFT JOIN customer_campaign_transactions t
    ON p.campaign_id = t.campaign_id AND p.customer_id = t.customer_id

LEFT JOIN customer_campaign_rewards r
    ON p.campaign_id = r.campaign_id AND p.customer_id = r.customer_id

WHERE p.eligible_flag = TRUE;

In [0]:
%sql
-- Check whether the number of customer stored at campaign level same as number independently calculated from customer level. 
-- Expected : Zero rows
WITH participants_from_customer_grain AS (
    SELECT
        campaign_id,
        COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END) AS participating_customers_check
    FROM `campaign&promotion`.gold.customer_campaign_performance
    GROUP BY campaign_id
)

SELECT
    cp.campaign_id,
    cp.participating_customers AS participating_customers_campaign_grain,
    pcg.participating_customers_check,
    cp.participating_customers - pcg.participating_customers_check AS participant_gap
FROM `campaign&promotion`.gold.campaign_performance cp
JOIN participants_from_customer_grain pcg
    ON cp.campaign_id = pcg.campaign_id
WHERE cp.participating_customers - pcg.participating_customers_check <> 0;

In [0]:
%sql
-- Segment Summary : Customer segment
CREATE OR REPLACE VIEW `campaign&promotion`.gold.segment_summary AS

SELECT
    customer_segment,
    COUNT(DISTINCT customer_id) AS eligible_customers,
    COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END) AS participating_customers,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END)
        / NULLIF(COUNT(DISTINCT customer_id), 0),
        2
    ) AS participation_rate_pct,

    SUM(transaction_count) AS total_transaction_count,
    SUM(transaction_value) AS total_transaction_value,
    ROUND(
        SUM(transaction_value) / NULLIF(COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END), 0),
        2
    ) AS avg_transaction_value_per_participant,

    SUM(reward_cost) AS total_reward_cost,
    COUNT(DISTINCT CASE WHEN is_new_customer AND participated_flag THEN customer_id END) AS new_customers_acquired

FROM `campaign&promotion`.gold.customer_campaign_performance
GROUP BY customer_segment;

In [0]:
%sql
SELECT * FROM `campaign&promotion`.gold.segment_summary
ORDER BY total_transaction_value DESC;

In [0]:
%sql
-- Segment summary: Customer type
SELECT
    customer_type,
    COUNT(DISTINCT customer_id) AS eligible_customers,
    COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END) AS participating_customers,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END)
        / NULLIF(COUNT(DISTINCT customer_id), 0),
        2
    ) AS participation_rate_pct,
    SUM(transaction_value) AS total_transaction_value,
    SUM(reward_cost) AS total_reward_cost
FROM `campaign&promotion`.gold.customer_campaign_performance
GROUP BY customer_type
ORDER BY total_transaction_value DESC;

In [0]:
%sql
-- Sgement Summary : Acquistion Channel
SELECT
    acquisition_channel,
    COUNT(DISTINCT customer_id) AS eligible_customers,
    COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END) AS participating_customers,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END)
        / NULLIF(COUNT(DISTINCT customer_id), 0),
        2
    ) AS participation_rate_pct,
    SUM(transaction_value) AS total_transaction_value,
    COUNT(DISTINCT CASE WHEN is_new_customer AND participated_flag THEN customer_id END) AS new_customers_acquired
FROM `campaign&promotion`.gold.customer_campaign_performance
GROUP BY acquisition_channel
ORDER BY participation_rate_pct DESC;

In [0]:
%sql
-- Segment Summary: Region
SELECT
    region,
    COUNT(DISTINCT customer_id) AS eligible_customers,
    COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END) AS participating_customers,
    ROUND(
        100.0 * COUNT(DISTINCT CASE WHEN participated_flag THEN customer_id END)
        / NULLIF(COUNT(DISTINCT customer_id), 0),
        2
    ) AS participation_rate_pct,
    SUM(transaction_value) AS total_transaction_value
FROM `campaign&promotion`.gold.customer_campaign_performance
GROUP BY region
ORDER BY total_transaction_value DESC;

### New vs. existing customers: which campaigns are most effective for each?


In [0]:
%sql
-- Top campaigns for NEW customers
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    COUNT(DISTINCT customer_id) AS new_participants,
    SUM(transaction_value) AS new_customer_transaction_value,
    SUM(reward_cost) AS new_customer_reward_cost
FROM `campaign&promotion`.gold.customer_campaign_performance
WHERE participated_flag = TRUE AND is_new_customer = TRUE
GROUP BY campaign_id, campaign_name, campaign_type
ORDER BY new_customer_transaction_value DESC
LIMIT 10;

In [0]:
%sql

-- Top campaigns for EXISTING customers
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    COUNT(DISTINCT customer_id) AS existing_participants,
    SUM(transaction_value) AS existing_customer_transaction_value,
    SUM(reward_cost) AS existing_customer_reward_cost
FROM `campaign&promotion`.gold.customer_campaign_performance
WHERE participated_flag = TRUE AND is_new_customer = FALSE
GROUP BY campaign_id, campaign_name, campaign_type
ORDER BY existing_customer_transaction_value DESC
LIMIT 10;

### Directional flag: reward spend on existing vs. new customers


In [0]:
%sql
SELECT
    is_new_customer,
    COUNT(DISTINCT customer_id) AS participants,
    SUM(reward_cost) AS total_reward_cost,
    ROUND(
        100.0 * SUM(reward_cost) / SUM(SUM(reward_cost)) OVER (),
        2
    ) AS pct_of_total_reward_cost,
    SUM(transaction_value) AS total_transaction_value
FROM `campaign&promotion`.gold.customer_campaign_performance
WHERE participated_flag = TRUE
GROUP BY is_new_customer;

In [0]:
%sql
SELECT
    campaign_type,
    is_new_customer,
    COUNT(DISTINCT customer_id) AS participants,
    SUM(reward_cost) AS total_reward_cost,
    SUM(transaction_value) AS total_transaction_value
FROM `campaign&promotion`.gold.customer_campaign_performance
WHERE participated_flag = TRUE
GROUP BY campaign_type, is_new_customer
ORDER BY campaign_type, is_new_customer DESC;